In [ ]:
import math

import matplotlib.pyplot as plt
import mujoco

import mujoco_toolbox as mjtb

In [ ]:
Vnom = 6
G = 55.5
t_stall = 15 / 100 / G
i_stall = 0.6
R = Vnom / i_stall

kt = t_stall / i_stall
ke = kt

V_control = 5

b_fit = 1.404e-6
kp_fit = 8.896

In [ ]:
def leg_controller(_: mujoco.MjModel, data: mujoco.MjData) -> None:
    """Simple leg controller for the MuJoCo model."""
    # Current state
    w = data.qvel[1]  # Current velocity of joint 1
    actual = data.qpos[1]  # Current position of joint 1

    desired = math.pi if data.time > 1 else 0

    error = desired - actual

    v = kp_fit * error

    v = max(min(v, V_control), -V_control)

    torque = (kt * (v - (ke * w * G)) / R - b_fit * w * G) * G

    data.ctrl[0] = torque  # control first joint

In [ ]:
rec = mjtb.Simulation(
    "../tests/models/box_and_leg.xml",
    initial_conditions={"qpos": [0, 0, 0]},
    controller=leg_controller,
).run(2, capture="all", render=True, fps=30, resolution=(640, 480))

In [ ]:
rec

In [ ]:
len(rec.frames)

In [ ]:
len(rec.data)

In [ ]:
rec.show()

In [ ]:
time_of_max_qpos = rec.time[rec.data["qpos"][:, 0].argmax()]

plt.imshow(rec.frame_at(time_of_max_qpos))
plt.title("Max Qpos")
plt.axis("off")
plt.show()

In [ ]:
plt.plot(rec.time, rec.data["qpos"][:, 0])
plt.show()